# Análisis Estadístico de Vectores Alpha (α₁-α₈) - Punto 2.5-2.6

**Objetivo:** Analizar la distribución y relevancia de los pesos α₁-α₈ asignados por el Algoritmo Genético a cada Índice de Daño (DI).

**Contexto:**
- **Vectores alpha:** Pesos normalizados (Σα = 1) asignados a cada DI por el AG
- **8 Índices de Daño (DI):**
  1. **DI1_COMAC:** Modal Assurance Criterion (MAC) nodal
  2. **DI2_COPERMOD:** Correlation of Mode Shapes
  3. **DI3_DR:** Damage Ratio
  4. **DI4_ECOPERMOD:** Enhanced COPERMOD
  5. **DI5_FDAC:** Frequency Domain Assurance Criterion
  6. **DI6_MSE:** Mean Squared Error
  7. **DI7_PDR:** Partial Damage Ratio
  8. **DI8_EFVI:** Energy Fraction Vibrational Indicator

**Preguntas de investigación:**
1. ¿Qué DIs son más relevantes (mayor peso promedio)?
2. ¿Los pesos varían según severidad del daño?
3. ¿Los pesos varían según tipo de elemento?
4. ¿Existe correlación entre alphas y el ICD?
5. ¿Los alphas convergen (poca varianza) o son inestables?

**Referencias:**
- MINUTA_SESION_2026-03-02.txt (Sección 2.5-2.6)
- todos_los_resultados.xlsx (columnas alpha1-alpha8)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficas
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("tab10")
%matplotlib inline

print("✓ Librerías cargadas")

## 1. Configuración y Carga de Datos

In [ ]:
# Rutas absolutas
base_path = Path.home() / 'github' / 'Proyecto-doctoral'
resultados_nuevos = base_path / 'outputs' / 'resultados_nuevos'

# Selección de tipo de daño
TIPO_DANO = 'abolladura'  # Cambiar a 'corrosion' para procesar corrosión

# Rutas según tipo de daño
if TIPO_DANO == 'abolladura':
    excel_path = base_path / 'Resultados' / 'abolladura_2026-02-26_05-26-02' / 'todos_los_resultados.xlsx'
    df_icd_path = resultados_nuevos / 'abolladuras_2026' / 'todos_los_resultados_con_ICD_abolladura.xlsx'
    output_dir = resultados_nuevos / 'abolladuras_2026'
elif TIPO_DANO == 'corrosion':
    excel_path = base_path / 'Resultados' / 'corrosion_2026-02-27_06-07-17' / 'todos_los_resultados.xlsx'
    df_icd_path = resultados_nuevos / 'corrosion_2026' / 'todos_los_resultados_con_ICD_corrosion.xlsx'
    output_dir = resultados_nuevos / 'corrosion_2026'
else:
    raise ValueError(f"Tipo de daño '{TIPO_DANO}' no reconocido")

# Nombres de los DIs
NOMBRES_DI = [
    'COMAC',
    'COPERMOD',
    'DR',
    'ECOPERMOD',
    'FDAC',
    'MSE',
    'PDR',
    'EFVI'
]

COLUMNAS_ALPHA = [f'alpha{i}' for i in range(1, 9)]

print(f"\n{'='*70}")
print(f"CONFIGURACIÓN - ANÁLISIS DE ALPHAS ({TIPO_DANO.upper()})")
print(f"{'='*70}")
print(f"Dataset original: {excel_path.name}")
print(f"Dataset con ICD: {df_icd_path.name}")
print(f"Columnas alpha: {COLUMNAS_ALPHA}")

In [ ]:
# Cargar dataset original con alphas
print("\n📂 Cargando dataset con vectores alpha...")
df = pd.read_excel(excel_path, engine='openpyxl')

# Verificar presencia de columnas alpha
missing_alphas = [col for col in COLUMNAS_ALPHA if col not in df.columns]
if missing_alphas:
    raise ValueError(f"Columnas alpha faltantes: {missing_alphas}")

print(f"✓ Dimensiones: {df.shape}")
print(f"✓ Columnas alpha presentes: {all(col in df.columns for col in COLUMNAS_ALPHA)}")

# Cargar dataset con ICD (si existe)
if df_icd_path.exists():
    print(f"\n📂 Cargando dataset con ICD...")
    df_icd = pd.read_excel(df_icd_path, engine='openpyxl')
    
    # Merge con alphas
    df = df.merge(df_icd[['ID', 'ICD']], on='ID', how='left')
    print(f"✓ ICD agregado al dataset")
else:
    print(f"\n⚠️  Dataset con ICD no encontrado. Ejecutar notebook 03_calculo_ICD.ipynb primero.")
    df['ICD'] = np.nan

# Mostrar primeras filas
print(f"\n📊 Primeras filas (con alphas):")
cols_mostrar = ['ID', 'Elemento', 'Porcentaje'] + COLUMNAS_ALPHA[:4] + ['ICD']
display(df[cols_mostrar].head())

## 2. Verificación de Normalización (Σα = 1)

In [ ]:
# Calcular suma de alphas por corrida
df['suma_alphas'] = df[COLUMNAS_ALPHA].sum(axis=1)

print(f"\n📊 Verificación de normalización (Σα):")
print(df['suma_alphas'].describe())

# Verificar si alguna corrida tiene suma ≠ 1 (tolerancia ±0.01)
no_normalizados = df[(df['suma_alphas'] < 0.99) | (df['suma_alphas'] > 1.01)]

if len(no_normalizados) > 0:
    print(f"\n⚠️  {len(no_normalizados)} corridas con Σα ≠ 1 (tolerancia ±0.01)")
    display(no_normalizados[['ID', 'Elemento', 'Porcentaje', 'suma_alphas']].head())
else:
    print(f"\n✓ Todas las corridas tienen Σα ≈ 1 (normalización correcta)")

# Gráfica de distribución
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['suma_alphas'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, label='Σα = 1 (esperado)')
ax.set_xlabel('Suma de alphas (Σα)', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title('Distribución de la Suma de Vectores Alpha', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / f'verificacion_normalizacion_alphas_{TIPO_DANO}.png', dpi=300)
plt.show()

## 3. Estadísticas Descriptivas por DI

In [ ]:
# Calcular estadísticas por DI
stats_alphas = df[COLUMNAS_ALPHA].describe().T
stats_alphas['DI'] = NOMBRES_DI
stats_alphas = stats_alphas[['DI', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats_alphas = stats_alphas.sort_values('mean', ascending=False)

print(f"\n📊 Estadísticas de vectores alpha por DI (ordenados por peso promedio):")
display(stats_alphas.style.format({
    'mean': '{:.4f}',
    'std': '{:.4f}',
    'min': '{:.4f}',
    '25%': '{:.4f}',
    '50%': '{:.4f}',
    '75%': '{:.4f}',
    'max': '{:.4f}'
}))

# Identificar top 3 DIs más influyentes
top3_dis = stats_alphas.head(3)['DI'].tolist()
print(f"\n🏆 Top 3 DIs más influyentes: {', '.join(top3_dis)}")

## 4. Visualización: Boxplots de Alpha por DI

In [ ]:
# Preparar datos para boxplot
df_alphas_melted = pd.melt(df[COLUMNAS_ALPHA], var_name='DI', value_name='Alpha')
df_alphas_melted['DI'] = df_alphas_melted['DI'].map(lambda x: NOMBRES_DI[int(x.replace('alpha', '')) - 1])

# Ordenar por peso promedio
orden_dis = stats_alphas['DI'].tolist()

# Gráfica
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_alphas_melted, x='DI', y='Alpha', order=orden_dis, ax=ax)
ax.set_xlabel('Índice de Daño (DI)', fontsize=12)
ax.set_ylabel('Peso α', fontsize=12)
ax.set_title(f'Distribución de Pesos Alpha por DI - {TIPO_DANO.capitalize()}', 
             fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(output_dir / f'boxplot_alphas_por_DI_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: boxplot_alphas_por_DI_{TIPO_DANO}.png")

## 5. Variación de Alphas vs Severidad del Daño

In [ ]:
# Calcular peso promedio por severidad para cada DI
alphas_por_severidad = df.groupby('Porcentaje')[COLUMNAS_ALPHA].mean()

# Gráfica
fig, ax = plt.subplots(figsize=(14, 7))

for i, alpha_col in enumerate(COLUMNAS_ALPHA):
    ax.plot(alphas_por_severidad.index, alphas_por_severidad[alpha_col], 
            marker='o', linewidth=2, markersize=6, label=NOMBRES_DI[i])

ax.set_xlabel('Severidad del daño (%)', fontsize=12)
ax.set_ylabel('Peso α promedio', fontsize=12)
ax.set_title(f'Evolución de Pesos Alpha vs Severidad - {TIPO_DANO.capitalize()}', 
             fontsize=14, fontweight='bold')
ax.legend(title='Índice de Daño', fontsize=9, title_fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / f'alphas_vs_severidad_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: alphas_vs_severidad_{TIPO_DANO}.png")

## 6. Heatmap: Correlación entre Alphas

In [ ]:
# Calcular matriz de correlación
corr_matrix = df[COLUMNAS_ALPHA].corr()

# Renombrar ejes con nombres de DIs
corr_matrix.index = NOMBRES_DI
corr_matrix.columns = NOMBRES_DI

# Gráfica
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title(f'Matriz de Correlación entre Pesos Alpha - {TIPO_DANO.capitalize()}', 
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(output_dir / f'correlacion_alphas_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: correlacion_alphas_{TIPO_DANO}.png")

# Identificar correlaciones más fuertes
corr_flat = corr_matrix.unstack()
corr_flat = corr_flat[corr_flat != 1.0].sort_values(ascending=False)

print(f"\n📊 Top 5 correlaciones más fuertes (positivas):")
for (di1, di2), corr in corr_flat.head(5).items():
    print(f"  {di1} ↔ {di2}: {corr:.3f}")

print(f"\n📊 Top 5 correlaciones más fuertes (negativas):")
for (di1, di2), corr in corr_flat.tail(5).items():
    print(f"  {di1} ↔ {di2}: {corr:.3f}")

## 7. Correlación entre Alphas e ICD

In [ ]:
# Solo ejecutar si ICD está disponible
if 'ICD' in df.columns and not df['ICD'].isna().all():
    # Calcular correlación de cada alpha con ICD
    corr_icd = df[COLUMNAS_ALPHA + ['ICD']].corr()['ICD'].drop('ICD').sort_values(ascending=False)
    corr_icd.index = NOMBRES_DI
    
    print(f"\n📊 Correlación entre Alphas e ICD:")
    for di, corr in corr_icd.items():
        print(f"  {di}: {corr:.4f}")
    
    # Gráfica de barras
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['green' if x > 0 else 'red' for x in corr_icd.values]
    ax.barh(corr_icd.index, corr_icd.values, color=colors, alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('Correlación de Pearson', fontsize=12)
    ax.set_ylabel('Índice de Daño (DI)', fontsize=12)
    ax.set_title(f'Correlación entre Pesos Alpha e ICD - {TIPO_DANO.capitalize()}', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(output_dir / f'correlacion_alphas_ICD_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Gráfica guardada: correlacion_alphas_ICD_{TIPO_DANO}.png")
    
    # Identificar DI con mayor/menor correlación con ICD
    di_max_corr = corr_icd.idxmax()
    di_min_corr = corr_icd.idxmin()
    print(f"\n🏆 DI con mayor correlación positiva con ICD: {di_max_corr} ({corr_icd[di_max_corr]:.3f})")
    print(f"📉 DI con mayor correlación negativa con ICD: {di_min_corr} ({corr_icd[di_min_corr]:.3f})")
    
else:
    print(f"\n⚠️  ICD no disponible. Ejecutar notebook 03_calculo_ICD.ipynb primero.")

## 8. Análisis de Estabilidad: Coeficiente de Variación

In [ ]:
# Calcular coeficiente de variación (CV = std/mean) para cada DI
cv_alphas = (df[COLUMNAS_ALPHA].std() / df[COLUMNAS_ALPHA].mean()).sort_values()
cv_alphas.index = [NOMBRES_DI[int(idx.replace('alpha', '')) - 1] for idx in cv_alphas.index]

print(f"\n📊 Coeficiente de Variación (CV) por DI:")
print("   (CV bajo → peso estable, CV alto → peso inestable)\n")
for di, cv in cv_alphas.items():
    estabilidad = 'Muy estable' if cv < 0.3 else 'Estable' if cv < 0.5 else 'Inestable' if cv < 1.0 else 'Muy inestable'
    print(f"  {di}: CV = {cv:.3f} ({estabilidad})")

# Gráfica de barras
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x < 0.5 else 'orange' if x < 1.0 else 'red' for x in cv_alphas.values]
ax.barh(cv_alphas.index, cv_alphas.values, color=colors, alpha=0.7, edgecolor='black')
ax.axvline(x=0.5, color='orange', linewidth=2, linestyle='--', label='CV = 0.5 (límite estabilidad)')
ax.axvline(x=1.0, color='red', linewidth=2, linestyle='--', label='CV = 1.0 (alta inestabilidad)')
ax.set_xlabel('Coeficiente de Variación (CV)', fontsize=12)
ax.set_ylabel('Índice de Daño (DI)', fontsize=12)
ax.set_title(f'Estabilidad de Pesos Alpha (CV) - {TIPO_DANO.capitalize()}', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(output_dir / f'estabilidad_alphas_CV_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica guardada: estabilidad_alphas_CV_{TIPO_DANO}.png")

## 9. Análisis por Tipo de Elemento (si aplica)

In [ ]:
# Verificar si existe columna de tipo de elemento
if 'Tipo_elemento_a_buscar' in df.columns:
    tipos_elementos = df['Tipo_elemento_a_buscar'].unique()
    
    print(f"\n📊 Peso promedio de alphas por tipo de elemento:")
    
    # Calcular peso promedio por tipo de elemento
    alphas_por_tipo = df.groupby('Tipo_elemento_a_buscar')[COLUMNAS_ALPHA].mean().T
    alphas_por_tipo.index = NOMBRES_DI
    
    display(alphas_por_tipo.style.format('{:.4f}'))
    
    # Gráfica de barras agrupadas
    fig, ax = plt.subplots(figsize=(14, 7))
    alphas_por_tipo.plot(kind='bar', ax=ax, width=0.8, edgecolor='black')
    ax.set_xlabel('Índice de Daño (DI)', fontsize=12)
    ax.set_ylabel('Peso α promedio', fontsize=12)
    ax.set_title(f'Pesos Alpha por Tipo de Elemento - {TIPO_DANO.capitalize()}', 
                 fontsize=14, fontweight='bold')
    ax.legend(title='Tipo de elemento', fontsize=10, title_fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(output_dir / f'alphas_por_tipo_elemento_{TIPO_DANO}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Gráfica guardada: alphas_por_tipo_elemento_{TIPO_DANO}.png")
    
else:
    print(f"\n⚠️  Columna 'Tipo_elemento_a_buscar' no encontrada. Saltando análisis por tipo.")

## 10. Resumen Ejecutivo

In [ ]:
print(f"\n{'='*70}")
print(f"RESUMEN EJECUTIVO - ANÁLISIS DE VECTORES ALPHA ({TIPO_DANO.upper()})")
print(f"{'='*70}")

print(f"\n1️⃣ DIs MÁS INFLUYENTES (top 3):")
for i, (di, weight) in enumerate(zip(stats_alphas.head(3)['DI'], stats_alphas.head(3)['mean']), 1):
    print(f"   {i}. {di}: α̅ = {weight:.4f}")

print(f"\n2️⃣ DIs MÁS ESTABLES (CV bajo):")
for i, (di, cv) in enumerate(cv_alphas.head(3).items(), 1):
    print(f"   {i}. {di}: CV = {cv:.3f}")

print(f"\n3️⃣ DIs MÁS INESTABLES (CV alto):")
for i, (di, cv) in enumerate(cv_alphas.tail(3).items(), 1):
    print(f"   {i}. {di}: CV = {cv:.3f}")

if 'ICD' in df.columns and not df['ICD'].isna().all():
    print(f"\n4️⃣ DIs CON MAYOR CORRELACIÓN CON ICD:")
    for i, (di, corr) in enumerate(corr_icd.head(3).items(), 1):
        print(f"   {i}. {di}: r = {corr:.3f}")

print(f"\n5️⃣ VERIFICACIÓN DE NORMALIZACIÓN:")
if len(no_normalizados) == 0:
    print(f"   ✓ Todas las corridas tienen Σα ≈ 1 (normalización correcta)")
else:
    print(f"   ⚠️  {len(no_normalizados)} corridas con Σα ≠ 1")

print(f"\n{'='*70}")
print("\n📁 ARCHIVOS GENERADOS:")
for file in sorted(output_dir.glob('*alphas*.png')):
    print(f"  - {file.name}")

print(f"\n{'='*70}")
print("✅ Análisis de vectores alpha completado exitosamente")
print(f"{'='*70}")

## 📝 Conclusiones y Próximos Pasos

**Conclusiones preliminares:**
1. Los pesos alpha revelan qué DIs son más relevantes para la detección de daños
2. El coeficiente de variación indica la estabilidad del AG en la asignación de pesos
3. Las correlaciones entre alphas muestran redundancia o complementariedad entre DIs
4. La correlación alpha-ICD identifica qué DIs contribuyen más a la calidad de detección

**Interpretación:**
- **Peso alto + CV bajo:** DI confiablemente importante (usar siempre)
- **Peso alto + CV alto:** DI importante pero inconsistente (requiere análisis adicional)
- **Peso bajo + CV bajo:** DI consistentemente poco importante (considerar eliminar)
- **Peso bajo + CV alto:** DI irrelevante e inestable (candidato a eliminación)

**Próximos pasos (Punto 2.7-3.0 del paper):**
1. Comparar ICD 2026 vs resultados previos (baseline)
2. Métricas de clasificación: TP, FP, TN, FN, Precision, Recall, F1-score
3. Tabla/figura de resultados por tipo de elemento y severidad
4. Análisis de sensibilidad: ¿Eliminar DIs con peso bajo mejora el rendimiento?
5. Validación cruzada: ¿Los pesos alpha se mantienen en subconjuntos de datos?